# embedding_text 설계 비교 (V1~V4) — Colab 단독 실행용

**목적**: `table_catalog.json`의 `embedding_text`를 어떻게 구성해야 검색 성능이 가장 좋은지
비교한다. 모델은 고정(`intfloat/multilingual-e5-large` + `BAAI/bge-reranker-v2-m3`)하고,
**표 설명 텍스트 설계 자체**만 4가지 버전으로 바꿔가며 같은 골든셋으로 평가한다.

## 비교 대상 4버전
- **V1**: title만
- **V2**: title + keywords
- **V3**: title + keywords + description (= 지금 프로덕션 `embedding_text`와 동일, 기준선)
- **V4**: V3 + 언론/기사체 표현(동의어, 실제 기사에서 확인된 표현만 선별 추가 — 54개 표 중
  근거 있는 19개 표에만 추가, 나머지는 V3과 동일하게 둠)

## 업로드해야 하는 파일 4개
왼쪽 사이드바 Files 패널의 업로드 아이콘으로 아래 4개를 `/content/`에 직접 올려두세요
(파일명 그대로 유지). 이번 라운드에서 골든셋을 46건(고신뢰 37건)으로 보강했으니 반드시
최신 버전을 올릴 것.

1. `table_catalog.json` (`agent/mapping/table_catalog.json`)
2. `embedding_text_versions.json` (`notebooks/embedding_text_versions.json` — V1~V4 텍스트가
   표별로 이미 계산돼 있는 파일)
3. `추출 골든셋 단위 분리.xlsx` (보강판, `notebooks/`)
4. `매핑 골든셋 ord 추가.xlsx` (보강판, `notebooks/`)

## 고정 모델
- 임베딩: `intfloat/multilingual-e5-large`
- 리랭커: `BAAI/bge-reranker-v2-m3`

각 버전마다 **임베딩 단독 성능**과 **임베딩 top-10 + 리랭커 재정렬 성능**을 둘 다 재서,
"리랭커가 약한 embedding_text를 얼마나 보완해주는지"까지 같이 본다. 추가로
`keyword_search`(모델 불필요, 규칙 기반) 성능도 같이 측정해서, 실패 사례가 키워드
사전의 한계인지 embedding_text 설계의 한계인지 구분할 근거로 쓴다.

**GPU**: A100 권장(요청하신 대로). T4에서도 동작은 하지만 느립니다.

**지표**: Top-1 정확도 / Recall@3 / Recall@5 / MRR / 평균 latency(ms).


In [ ]:
# torch는 코랩에 이미 CUDA 지원 버전이 깔려 있어 재설치하지 않는다.
!pip install -q sentence-transformers transformers accelerate pandas openpyxl

## 1. 데이터 파일 확인

파일 4개는 왼쪽 Files 패널에서 미리 업로드해뒀다는 전제. 아래 셀은 그 파일들이
`/content/`(코랩) 혹은 현재 폴더(로컬)에 실제로 있는지만 확인한다.

In [ ]:
from pathlib import Path

data_dir = Path.cwd()

REQUIRED_FILES = [
    "table_catalog.json",
    "embedding_text_versions.json",
    "추출 골든셋 단위 분리.xlsx",
    "매핑 골든셋 ord 추가.xlsx",
]

missing = [name for name in REQUIRED_FILES if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        f"다음 파일이 {data_dir}에 없습니다: {missing}. "
        f"왼쪽 Files 패널의 업로드 아이콘으로 파일명 그대로 올린 뒤 이 셀을 다시 실행하세요."
    )

CATALOG_PATH = data_dir / "table_catalog.json"
VERSIONS_PATH = data_dir / "embedding_text_versions.json"
CLAIMS_XLSX = data_dir / "추출 골든셋 단위 분리.xlsx"
MAPPING_XLSX = data_dir / "매핑 골든셋 ord 추가.xlsx"

print("파일 확인 완료:")
for p in [CATALOG_PATH, VERSIONS_PATH, CLAIMS_XLSX, MAPPING_XLSX]:
    print(f"  {p.name} ({p.stat().st_size:,} bytes)")

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
else:
    print(
        "GPU가 잡히지 않았습니다. 런타임 > 런타임 유형 변경 > GPU(A100)로 바꾼 뒤 "
        "런타임을 다시 시작하세요."
    )

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2. 골든셋 로더 (`agent/mapping/golden_set.py` 이식)

이번에 보강된 골든셋(46건, 고신뢰 37건)을 로드한다. 로직은 이전 노트북들과 동일 —
`match_status`가 "매칭 실패"/"미완료"인 claim 제외, `TABLE_ID_OVERRIDES`로 원본 tblId를
카탈로그 실제값으로 보정.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Optional

import pandas as pd

TABLE_ID_OVERRIDES: dict[str, str] = {
    "DT_1K41013": "DT_1K41017",
    "DT_1G18011": "DT_1G18007",
    "DT_1L9U001": "DT_1L9U103",
    "DT_1HDALF05": "DT_1L9U103",
    "DT_1DA7E33S": "DT_1DA7E06S_NEW",
    "DT_1DA7002S": "DT_1DA7102S",
}

_NO_ANSWER_STATUSES = {"매칭 실패", "미완료"}


@dataclass
class GoldExample:
    claim_id: str
    sentence: str
    gold_table_id: str
    raw_table_id: str
    match_status: str
    confidence: str
    overridden: bool


def _confidence_of(match_status: str) -> str:
    return "high" if match_status.startswith("값 확인됨") else "low"


def load_golden_set(
    *, claims_path, mapping_path, catalog_path, warn_missing: bool = True
) -> list[GoldExample]:
    claims_df = pd.read_excel(claims_path)
    mapping_df = pd.read_excel(mapping_path)
    merged = claims_df.merge(mapping_df, on="claim_id", how="left")

    catalog = json.loads(Path(catalog_path).read_text(encoding="utf-8"))
    catalog_ids = {t["tblId"] for t in catalog["tables"]}

    examples: list[GoldExample] = []
    skipped_no_answer = 0
    skipped_missing_in_catalog: list[tuple[str, str]] = []

    for _, row in merged.iterrows():
        status = row.get("match_status")
        raw_id = row.get("kosis_table_id")

        if pd.isna(raw_id) or status in _NO_ANSWER_STATUSES:
            skipped_no_answer += 1
            continue

        gold_id = TABLE_ID_OVERRIDES.get(raw_id, raw_id)
        if gold_id not in catalog_ids:
            skipped_missing_in_catalog.append((row["claim_id"], gold_id))
            continue

        examples.append(
            GoldExample(
                claim_id=row["claim_id"],
                sentence=row["claim_sentence"],
                gold_table_id=gold_id,
                raw_table_id=raw_id,
                match_status=status,
                confidence=_confidence_of(status),
                overridden=(gold_id != raw_id),
            )
        )

    if warn_missing:
        print(
            f"[golden_set] 총 {len(merged)}건 중 정답 없음/미완료 {skipped_no_answer}건 제외, "
            f"카탈로그에 없는 gold_table_id {len(skipped_missing_in_catalog)}건 제외 "
            f"-> 평가 가능 {len(examples)}건"
        )
        if skipped_missing_in_catalog:
            print(f"  카탈로그에 없는 항목: {skipped_missing_in_catalog}")

    return examples


examples = load_golden_set(
    claims_path=CLAIMS_XLSX, mapping_path=MAPPING_XLSX, catalog_path=CATALOG_PATH
)
n_high = sum(1 for e in examples if e.confidence == "high")
print(f"평가 가능 claim: {len(examples)}건 (high-confidence {n_high}건)")

## 3. 카탈로그 + embedding_text 버전 로드

In [ ]:
catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
catalog_ids = [t["tblId"] for t in catalog["tables"]]

versions_data = json.loads(VERSIONS_PATH.read_text(encoding="utf-8"))
version_tables = versions_data["tables"]

VERSION_KEYS = {
    "V1(title만)": "v1_title",
    "V2(title+keywords)": "v2_title_keywords",
    "V3(title+keywords+description)": "v3_title_keywords_description",
    "V4(V3+기사체 표현)": "v4_plus_press_expr",
}

print(f"카탈로그 표 개수: {len(catalog_ids)}")
print(f"V4에 추가 표현이 들어간 표 개수: {sum(1 for v in version_tables.values() if v['v4_added_terms'])}")

## 4. 평가 지표 (`agent/mapping/eval_metrics.py` 이식)

In [ ]:
import time
from dataclasses import field
from typing import Callable

DEFAULT_K_LIST = (1, 3, 5)


@dataclass
class EvalResult:
    model: str
    n: int
    recall_at_k: dict[int, float]
    mrr: float
    mean_latency_ms: float
    param_count: Optional[str] = None
    per_example: list[dict] = field(default_factory=list)

    @property
    def accuracy(self) -> float:
        """top-1 정확도 = Recall@1."""
        return self.recall_at_k.get(1, 0.0)


def rank_metrics(ranked_ids, gold_id, k_list=DEFAULT_K_LIST):
    recall = {k: gold_id in ranked_ids[:k] for k in k_list}
    rr = 1.0 / (ranked_ids.index(gold_id) + 1) if gold_id in ranked_ids else 0.0
    return recall, rr


def evaluate(
    model_name: str,
    rank_fn: Callable[[str], list[str]],
    examples: list[GoldExample],
    *,
    k_list=DEFAULT_K_LIST,
    param_count: Optional[str] = None,
) -> EvalResult:
    correct_at_k = {k: 0 for k in k_list}
    total_rr = 0.0
    latencies: list[float] = []
    per_example: list[dict] = []

    for ex in examples:
        t0 = time.perf_counter()
        ranked = rank_fn(ex.sentence)
        latencies.append((time.perf_counter() - t0) * 1000)

        recall, rr = rank_metrics(ranked, ex.gold_table_id, k_list)
        for k in k_list:
            correct_at_k[k] += int(recall[k])
        total_rr += rr

        per_example.append(
            {
                "claim_id": ex.claim_id,
                "gold_table_id": ex.gold_table_id,
                "confidence": ex.confidence,
                "rank": ranked.index(ex.gold_table_id) + 1 if ex.gold_table_id in ranked else None,
                "top1": ranked[0] if ranked else None,
            }
        )

    n = len(examples)
    return EvalResult(
        model=model_name,
        n=n,
        recall_at_k={k: correct_at_k[k] / n for k in k_list},
        mrr=total_rr / n,
        mean_latency_ms=sum(latencies) / n,
        param_count=param_count,
        per_example=per_example,
    )


def summarize(results: list[EvalResult], k_list=DEFAULT_K_LIST) -> pd.DataFrame:
    """결과 리스트를 발표 슬라이드용 DataFrame으로 정리."""
    rows = []
    for r in results:
        row = {"버전": r.model, "표본수": r.n, "top-1 정확도": f"{r.accuracy * 100:.1f}%"}
        for k in k_list:
            if k == 1:
                continue
            row[f"Recall@{k}"] = f"{r.recall_at_k[k] * 100:.1f}%"
        row["MRR"] = f"{r.mrr:.3f}"
        row["평균 latency(ms)"] = f"{r.mean_latency_ms:.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

## 5. 키워드 검색 (`agent/mapping/keyword_search.py` 이식, 베이스라인)

모델과 무관한 규칙 기반 검색. embedding_text 설계와 상관없이 항상 같은 성능이 나오므로,
"embedding_text를 아무리 잘 설계해도 키워드 검색보다 못하다면 애초에 키워드 사전 문제"
같은 판단의 기준선이 된다.

In [ ]:
import re
from dataclasses import field as _field


@dataclass
class Claim:
    sentence: str
    claim_type: str
    period: Optional[str] = None
    unit: Optional[str] = None
    population: Optional[str] = None


@dataclass
class TableCandidate:
    table_id: str
    table_name: str
    score: float
    required_slots: list = _field(default_factory=list)
    source_meta: Optional[str] = None


SYNONYMS: dict[str, list[str]] = {
    "취업자": ["취업자수"],
    "실업자": ["실업률"],
    "고용": ["고용률", "고용동향"],
    "일자리": ["고용률", "취업자수"],
    "장바구니": ["장바구니 물가", "소비자물가지수"],
    "집값": ["집값", "아파트 매매가격", "주택매매가격지수"],
    "부동산": ["주택매매가격지수", "주택가격동향"],
    "출산": ["합계출산율", "출생률", "저출산", "출생아수"],
    "저출생": ["저출산", "출생아수", "출생률"],
    "출생아": ["출생아수", "합계출산율", "출생률"],
    "인구감소": ["인구감소", "주민등록인구"],
    "수출이": ["수출액", "수출 증가율"],
    "수출은": ["수출액", "수출 증가율"],
    "수입": ["수입액"],
    "무역흑자": ["무역수지", "무역흑자"],
    "성장률": ["경제성장률", "GDP", "국내총생산"],
    "GDP": ["국내총생산", "경제성장률"],
    "이자율": ["대출금리", "예금은행 금리"],
    "기준금리": ["대출금리", "예금은행 금리"],
    "대출이자": ["대출금리", "여신금리"],
    "주가": ["코스피지수", "증시", "주가지수"],
    "코스피지수": ["코스피", "KOSPI"],
    "가계빚": ["가계신용", "가계부채"],
    "가계대출": ["가계신용", "가계부채"],
    "전셋값": ["전세가격", "전세지수"],
    "전세값": ["전세가격", "전세지수"],
    "이사": ["국내이동", "인구이동"],
    "학원비용": ["사교육비", "학원비"],
    "임금격차": ["시간당임금", "임금격차"],
}


def _normalize(text: str) -> str:
    return re.sub(r"\s+", "", text)


def _expand_query_terms(sentence: str) -> set[str]:
    terms: set[str] = set()
    normalized_sentence = _normalize(sentence)
    for raw_term, mapped_terms in SYNONYMS.items():
        if _normalize(raw_term) in normalized_sentence:
            terms.update(mapped_terms)
    return terms


def _score_table(sentence: str, expanded_terms: set[str], table: dict) -> tuple[float, list[str]]:
    matched: list[str] = []
    keywords = table.get("keywords", [])
    normalized_sentence = _normalize(sentence)

    for kw in keywords:
        if _normalize(kw) in normalized_sentence:
            matched.append(kw)
        elif kw in expanded_terms:
            matched.append(kw)

    if table.get("title", "") and re.search(re.escape(table["title"][:6]), sentence):
        matched.append(f"[title]{table['title']}")

    if not matched:
        return 0.0, []

    score = min(1.0, len(matched) / 3)
    return score, matched


def keyword_search(claim: Claim, *, top_k: int = 10, tables: list[dict] | None = None) -> list[TableCandidate]:
    tables = tables if tables is not None else catalog["tables"]
    expanded_terms = _expand_query_terms(claim.sentence)

    candidates: list[TableCandidate] = []
    for table in tables:
        score, matched = _score_table(claim.sentence, expanded_terms, table)
        if score <= 0:
            continue
        candidates.append(
            TableCandidate(
                table_id=table["tblId"],
                table_name=table["title"],
                score=score,
                required_slots=table.get("required_slots", []),
                source_meta=f"keyword_search matched={matched}",
            )
        )

    candidates.sort(key=lambda c: c.score, reverse=True)
    return candidates[:top_k]


def keyword_rank_fn(sentence: str, top_k: int = 5) -> list[str]:
    claim = Claim(sentence=sentence, claim_type="규모")
    return [c.table_id for c in keyword_search(claim, top_k=top_k)]


keyword_result = evaluate("keyword_search(베이스라인)", keyword_rank_fn, examples, param_count="0")
print(
    f"keyword_search: top-1={keyword_result.accuracy:.1%}  Recall@5={keyword_result.recall_at_k[5]:.1%}  "
    f"MRR={keyword_result.mrr:.3f}"
)

## 6. 임베딩(e5) 로더 — 버전별 재사용

`multilingual-e5-large`는 쿼리/문서를 비대칭으로 인코딩해야 한다(`query: `/`passage: `
프리픽스). 버전마다 문서(표) 쪽 텍스트만 바뀌고 쿼리(claim 문장) 인코딩 방식은 동일하다.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import gc

EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

# sentence-transformers의 CrossEncoder.predict() 래퍼가 일부 transformers 버전 조합에서
# "AttributeError: 'ne'" 류의 내부 배치 처리 버그를 일으켜서, BAAI 공식 모델 카드가 권장하는
# 방식(토크나이저 + AutoModelForSequenceClassification 직접 호출)으로 우회한다.
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL, trust_remote_code=True
).to(DEVICE).eval()
if DEVICE == "cuda":
    # A100은 bf16을 지원하지만, 방어적으로 fp16 고정(운영 코드와 동일 정책).
    reranker_model = reranker_model.half()


def embed_doc_texts(texts: list[str]) -> np.ndarray:
    inputs = [f"passage: {t}" for t in texts]
    return np.array(embed_model.encode(inputs, convert_to_numpy=True, show_progress_bar=False))


def embed_query(sentence: str) -> np.ndarray:
    return np.array(embed_model.encode([f"query: {sentence}"], convert_to_numpy=True, show_progress_bar=False)[0])


def cosine_topk(query_vec: np.ndarray, doc_matrix: np.ndarray, ids: list[str], top_k: int) -> list[tuple[str, float]]:
    q = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    d = doc_matrix / (np.linalg.norm(doc_matrix, axis=1, keepdims=True) + 1e-8)
    sims = d @ q
    order = np.argsort(-sims)[:top_k]
    return [(ids[i], float(sims[i])) for i in order]


def rerank_pool(sentence: str, pool_ids: list[str], doc_text_map: dict[str, str], top_k: int = 5) -> list[str]:
    if not pool_ids:
        return []
    docs = [doc_text_map[tid] for tid in pool_ids]
    pairs = [[sentence, doc] for doc in docs]
    with torch.no_grad():
        inputs = reranker_tokenizer(
            pairs, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(DEVICE)
        scores = reranker_model(**inputs).logits.view(-1).float().cpu().numpy()
    ranked = sorted(zip(pool_ids, scores), key=lambda pair: pair[1], reverse=True)
    return [tid for tid, _ in ranked][:top_k]


## 7. 버전별 평가 루프

각 버전(V1~V4)마다: (a) 표 54개를 그 버전 텍스트로 인코딩 → (b) 임베딩 단독 top-5 평가 →
(c) 임베딩 top-10 후보를 리랭커로 재정렬한 top-5 평가, 두 가지를 모두 잰다.

In [ ]:
all_results: dict[str, EvalResult] = {"keyword_search(베이스라인)": keyword_result}

for version_label, version_key in VERSION_KEYS.items():
    print(f"\n########## {version_label} ##########")
    doc_texts = [version_tables[tid][version_key] for tid in catalog_ids]
    doc_text_map = dict(zip(catalog_ids, doc_texts))
    doc_vecs = embed_doc_texts(doc_texts)

    def embed_rank_fn(sentence: str, _doc_vecs=doc_vecs, top_k: int = 5) -> list[str]:
        q_vec = embed_query(sentence)
        return [tid for tid, _ in cosine_topk(q_vec, _doc_vecs, catalog_ids, top_k)]

    embed_result = evaluate(f"{version_label} [임베딩 단독]", embed_rank_fn, examples)
    all_results[embed_result.model] = embed_result
    print(
        f"  [임베딩 단독] top-1={embed_result.accuracy:.1%}  Recall@5={embed_result.recall_at_k[5]:.1%}  "
        f"MRR={embed_result.mrr:.3f}  latency={embed_result.mean_latency_ms:.1f}ms"
    )

    def embed_rerank_fn(sentence: str, _doc_vecs=doc_vecs, _doc_text_map=doc_text_map, top_k: int = 5) -> list[str]:
        q_vec = embed_query(sentence)
        pool = [tid for tid, _ in cosine_topk(q_vec, _doc_vecs, catalog_ids, 10)]
        return rerank_pool(sentence, pool, _doc_text_map, top_k=top_k)

    rerank_result = evaluate(f"{version_label} [+리랭커]", embed_rerank_fn, examples)
    all_results[rerank_result.model] = rerank_result
    print(
        f"  [+리랭커]    top-1={rerank_result.accuracy:.1%}  Recall@5={rerank_result.recall_at_k[5]:.1%}  "
        f"MRR={rerank_result.mrr:.3f}  latency={rerank_result.mean_latency_ms:.1f}ms"
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summarize(list(all_results.values()))

## 8. high-confidence만 재검증

low-confidence(표 후보 확인 등) claim이 섞이면 정답 자체가 불확실해 버전 비교가 왜곡될
수 있다. high-confidence(37건)만 따로 다시 재서 순위가 바뀌는지 확인한다.

In [ ]:
high_conf_examples = [e for e in examples if e.confidence == "high"]
print(f"high-confidence 표본: {len(high_conf_examples)}건")

high_conf_results: dict[str, EvalResult] = {}

kw_high = evaluate("keyword_search(베이스라인)", keyword_rank_fn, high_conf_examples, param_count="0")
high_conf_results[kw_high.model] = kw_high

for version_label, version_key in VERSION_KEYS.items():
    doc_texts = [version_tables[tid][version_key] for tid in catalog_ids]
    doc_text_map = dict(zip(catalog_ids, doc_texts))
    doc_vecs = embed_doc_texts(doc_texts)

    def embed_rank_fn(sentence: str, _doc_vecs=doc_vecs, top_k: int = 5) -> list[str]:
        q_vec = embed_query(sentence)
        return [tid for tid, _ in cosine_topk(q_vec, _doc_vecs, catalog_ids, top_k)]

    def embed_rerank_fn(sentence: str, _doc_vecs=doc_vecs, _doc_text_map=doc_text_map, top_k: int = 5) -> list[str]:
        q_vec = embed_query(sentence)
        pool = [tid for tid, _ in cosine_topk(q_vec, _doc_vecs, catalog_ids, 10)]
        return rerank_pool(sentence, pool, _doc_text_map, top_k=top_k)

    r1 = evaluate(f"{version_label} [임베딩 단독]", embed_rank_fn, high_conf_examples)
    r2 = evaluate(f"{version_label} [+리랭커]", embed_rerank_fn, high_conf_examples)
    high_conf_results[r1.model] = r1
    high_conf_results[r2.model] = r2

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summarize(list(high_conf_results.values()))

## 9. 실패 사례 상세 (성공/실패 분석용)

버전별 `per_example` 기록에서 V3(현재 프로덕션 기준선)와 V4를 비교해, V4가 V3를 못 따라오는
claim / V4가 V3를 이기는 claim을 각각 뽑는다. 최종 리포트의 "성공·실패 사례 분석",
"keyword 부족 vs embedding_text 부족" 판단에 이 표를 근거로 쓴다.

In [ ]:
v3_embed = all_results["V3(title+keywords+description) [임베딩 단독]"]
v4_embed = all_results["V4(V3+기사체 표현) [임베딩 단독]"]

v3_by_claim = {p["claim_id"]: p for p in v3_embed.per_example}
v4_by_claim = {p["claim_id"]: p for p in v4_embed.per_example}
kw_by_claim = {p["claim_id"]: p for p in keyword_result.per_example}

rows = []
for ex in examples:
    v3p, v4p, kwp = v3_by_claim[ex.claim_id], v4_by_claim[ex.claim_id], kw_by_claim[ex.claim_id]
    rows.append(
        {
            "claim_id": ex.claim_id,
            "confidence": ex.confidence,
            "gold_table_id": ex.gold_table_id,
            "sentence": ex.sentence[:60],
            "keyword_rank": kwp["rank"],
            "V3_rank": v3p["rank"],
            "V4_rank": v4p["rank"],
            "V4_vs_V3": "개선" if (v4p["rank"] or 99) < (v3p["rank"] or 99) else (
                "악화" if (v4p["rank"] or 99) > (v3p["rank"] or 99) else "동일"
            ),
        }
    )

diff_df = pd.DataFrame(rows)
print("V4가 V3보다 개선/악화/동일 건수:")
print(diff_df["V4_vs_V3"].value_counts())
diff_df.sort_values(["V4_vs_V3", "confidence"])

## 10. 결과 다운로드

코랩 세션은 끝나면 사라지므로, 이 노트북에서 만든 표는 로컬 저장소로 직접 가져가야 한다.

In [ ]:
summarize(list(all_results.values())).to_csv("embedding_text_comparison_all.csv", index=False, encoding="utf-8-sig")
summarize(list(high_conf_results.values())).to_csv("embedding_text_comparison_high_conf.csv", index=False, encoding="utf-8-sig")
diff_df.to_csv("embedding_text_comparison_v3_vs_v4_diff.csv", index=False, encoding="utf-8-sig")

try:
    from google.colab import files

    files.download("embedding_text_comparison_all.csv")
    files.download("embedding_text_comparison_high_conf.csv")
    files.download("embedding_text_comparison_v3_vs_v4_diff.csv")
except ImportError:
    pass  # 로컬 실행: 다운로드 없이 현재 폴더에 CSV만 남긴다

print("저장 완료: embedding_text_comparison_all.csv, embedding_text_comparison_high_conf.csv, embedding_text_comparison_v3_vs_v4_diff.csv")

## 다음 단계

1. 다운로드한 CSV 3개를 저장소의 `notebooks/` 아래에 두거나, Claude Code 세션에 다시
   붙여넣어서 최종 분석 리포트(버전별 비교표/최적 설계/장단점/성공-실패 사례/keyword vs
   embedding_text 원인분석)를 작성할 것.
2. 승자 버전이 확정되면 `agent/mapping/table_catalog.json`의 `embedding_text` 필드를
   해당 버전 텍스트로 교체하고 `embedding_text_rule`(`_meta`)도 갱신할 것.
3. `table_embeddings_cache.json`은 `embedding_text`가 바뀌면 `embedding_search.py`가
   자동으로 재생성하므로 별도 조치 불필요.